## Import Libraries

In [72]:
import os, re, string, unicodedata
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from tqdm import tqdm
import spacy
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

## Setup

In [73]:
# Set default renderer for Plotly to VSCode
pio.renderers.default = "vscode"

# Show progress bars for pandas operations
tqdm.pandas()

In [74]:
# Ensure spaCy model is available
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    
    !python -m spacy download en_core_web_sm
    nlp = spacy.load("en_core_web_sm")

## Loading dataset

In [75]:
imdb_data = pd.read_csv('/Users/subhadeepdebnath/Developer/genai/experiments/nlp_playground/data/IMDB_Dataset.csv')
imdb_data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [76]:
print(f"Dataset shape: {imdb_data.shape}")
print(f"Dataset columns: {imdb_data.columns.tolist()}")
print("", imdb_data.info())

Dataset shape: (50000, 2)
Dataset columns: ['review', 'sentiment']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
 None


In [77]:
SAMPLE_SIZE = None        # Set to None to use full dataset
RANDOM_STATE = 42      # Random state for reproducibility

In [78]:
imdb_data["review"] = imdb_data["review"].astype(str)
imdb_data["sentiment"] = imdb_data["sentiment"].astype(str)

In [79]:
vc = (imdb_data["sentiment"]
      .value_counts()
      .rename_axis("sentiment")
      .reset_index(name="count"))

fig = px.bar(
    vc, x="sentiment", y="count",
    labels={"sentiment":"Sentiment", "count":"Count"},
    title="IMDb Sentiment Distribution"
)
fig.update_layout(yaxis=dict(gridcolor="rgba(0,0,0,0.1)"))

fig.show()

# Review length (in tokens) — before cleaning
imdb_data["review_len"] = imdb_data["review"].str.split().apply(len)
fig = px.histogram(
    imdb_data, x="review_len", nbins=60,
    title="Distribution of Raw Review Lengths (Tokens)"
)
fig.update_layout(bargap=0.02)
fig.show()


### Data Cleaning

In [80]:
def data_cleaning(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.lower()
    # keep only letters + apostrophes to preserve contractions
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

if imdb_data.get("clean") is None:
    imdb_data["clean"] = imdb_data["review"].progress_apply(data_cleaning)
    
y = imdb_data["sentiment"].map({"positive":1, "negative":0}).values


100%|██████████| 50000/50000 [00:02<00:00, 18482.73it/s]


## Data Preparation

In [81]:
def spacy_tokens(text: str):
    doc = nlp(text)
    # keep alpha tokens, remove stopwords, lemmatize
    return [t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop]

if "tokens" not in imdb_data.columns:
    imdb_data["tokens"] = imdb_data["clean"].progress_apply(spacy_tokens)
    imdb_data[["review","clean","tokens"]].head(2)

100%|██████████| 50000/50000 [29:43<00:00, 28.04it/s]  


## Feature Creation 

### Word2vec Vectorizer

In [82]:
sentences = imdb_data["tokens"].tolist()

w2v = Word2Vec(
    sentences=sentences,
    vector_size=100, window=5, min_count=5, sg=1, epochs=10, workers=4, seed=RANDOM_STATE
)
print("Word2Vec vocab size:", len(w2v.wv))
print("Sample words in vocab:", w2v)

# Save model
os.makedirs("models", exist_ok=True)
w2v.save("models/word2vec_imdb.model")


Word2Vec vocab size: 31040
Sample words in vocab: Word2Vec<vocab=31040, vector_size=100, alpha=0.025>


In [83]:
print(len(w2v.wv))             # Number of words in vocab
print(w2v.vector_size)         # Embedding dimensions (100)
print(w2v.wv.index_to_key[:5]) # Top 5 most frequent words
print(w2v.wv['good'][:10])     # First 10 dimensions of 'good'’s vector
print(w2v.wv.most_similar('good'))


31040
100
['movie', 'film', 'like', 'good', 'time']
[-0.02545915  0.19802344 -0.15343176  0.01600172 -0.3681397   0.03000967
 -0.22641955  0.20479332  0.07980295 -0.04400933]
[('great', 0.8097290396690369), ('decent', 0.7836442589759827), ('well', 0.770755410194397), ('excellent', 0.755618155002594), ('persbrandt', 0.7481318712234497), ('duller', 0.7461293935775757), ('kamerling', 0.7413848638534546), ('bad', 0.7368460297584534), ('think', 0.7180137038230896), ('okay', 0.7104043364524841)]


In [84]:
def avg_w2v(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0)

X_vec = np.vstack([avg_w2v(toks, w2v) for toks in imdb_data["tokens"]])
print("Feature matrix:", X_vec.shape)


Feature matrix: (50000, 100)


In [85]:
# Check word relationships
print("\nMost similar to 'good':", w2v.wv.most_similar("good", topn=5))
print("Similarity(good, bad):", w2v.wv.similarity("good", "bad"))
print("Similarity(movie, film):", w2v.wv.similarity("movie", "film"))



Most similar to 'good': [('great', 0.8097290396690369), ('decent', 0.7836442589759827), ('well', 0.770755410194397), ('excellent', 0.755618155002594), ('persbrandt', 0.7481318712234497)]
Similarity(good, bad): 0.7368461
Similarity(movie, film): 0.8534863


In [86]:
# Pick key sentiment words
words = ["good", "bad", "great", "terrible", "boring", "funny", "movie", "film", "plot", "acting"]
word_vecs = np.array([w2v.wv[w] for w in words if w in w2v.wv])
pca = PCA(n_components=2)
coords = pca.fit_transform(word_vecs)

fig = px.scatter(
    x=coords[:,0], y=coords[:,1], text=words,
    title="Semantic Word Space (Word2Vec + PCA)"
)
fig.update_traces(textposition="top center")
fig.show()


## Modelling


### Train-Test Split

In [87]:
X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42, stratify=y)

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape) 

Train shape: (40000, 100) | Test shape: (10000, 100)


### Text Classification

#### Logistic Regression

In [88]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Linear SVM": LinearSVC(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "Gaussian NB": GaussianNB()
}

In [89]:

results = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1  = f1_score(y_test, pred)
    results.append((name, acc, f1))
    print(f"\n📊 {name}")
    print(classification_report(y_test, pred, digits=3))

results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1"])
results_df.sort_values(by="Accuracy", ascending=False, inplace=True)
results_df


📊 Logistic Regression
              precision    recall  f1-score   support

           0      0.870     0.870     0.870      5000
           1      0.870     0.870     0.870      5000

    accuracy                          0.870     10000
   macro avg      0.870     0.870     0.870     10000
weighted avg      0.870     0.870     0.870     10000


📊 Linear SVM
              precision    recall  f1-score   support

           0      0.873     0.869     0.871      5000
           1      0.870     0.874     0.872      5000

    accuracy                          0.872     10000
   macro avg      0.872     0.872     0.872     10000
weighted avg      0.872     0.872     0.872     10000


📊 Random Forest
              precision    recall  f1-score   support

           0      0.868     0.817     0.842      5000
           1      0.827     0.876     0.851      5000

    accuracy                          0.847     10000
   macro avg      0.848     0.847     0.846     10000
weighted avg      0.

,Model,Accuracy,F1
1,Linear SVM,0.8716,0.871907
0,Logistic Regression,0.8703,0.870287
2,Random Forest,0.8465,0.850841
3,Gaussian NB,0.7791,0.776846


**Insight:**  
Unlike TF-IDF (which counts words), Word2Vec captures relationships between words.  
For instance, “excellent” and “great” have high cosine similarity even if they never co-occur.

✅ Classical NLP → Sparse Frequency Vectors  
✅ Hybrid NLP → Dense Semantic Embeddings  
✅ Deep NLP → Contextualized Word Representations (Transformers)


In [90]:
positive_words = ["good", "excellent", "amazing", "love", "enjoyed"]
negative_words = ["bad", "terrible", "boring", "hate", "worst"]

words = positive_words + negative_words
vecs = np.array([w2v_model.wv[w] for w in words if w in w2v_model.wv])
pca = PCA(n_components=2)
coords = pca.fit_transform(vecs)

colors = ["green"]*len(positive_words) + ["red"]*len(negative_words)

fig = px.scatter(
    x=coords[:,0], y=coords[:,1], text=words, color=colors,
    title="Sentiment Word Clusters (Word2Vec Semantic Space)"
)
fig.update_traces(textposition="top center")
fig.show()
